# Apache Hive on Google Colab — Apache Log File Analysis
### Big Data Analytics · Session 7 · Hands-on Session: Coding
**Learning outcome:** *Ability to perform analysis using Hive.*

This notebook installs **real Apache Hive** in your Colab session and analyses a web-server log file
using **HiveQL** — the SQL-like language Hive uses.

**Why Colab:** Colab runs Linux in the cloud, so nothing installs on your own laptop.
Mac and Windows students get an identical environment. You only need a browser.

---

### How to run
1. **Runtime → Run all** (or run cells top to bottom).
2. Section 1 takes **3–6 minutes** — it downloads Hadoop and Hive. Run it once and let it finish.
3. After that, every query returns in seconds.

> ⚠️ **Colab sessions are temporary.** If the runtime disconnects, re-run Section 1.
> Nothing is saved to your Google Drive unless you deliberately save it.


---
## Section 1 · Install Hive (run once per session)

Hive is not a standalone program — it sits on top of Hadoop and needs **Java**.
So we install three things in order: **Java → Hadoop → Hive**.

We pin exact versions and download from `archive.apache.org`, which keeps every past release
permanently. (The `downloads.apache.org` mirror deletes old versions, which is the usual reason
a tutorial's link suddenly dies mid-semester.)


### 1.1 · Java 8

Hive 3.x is built for **Java 8**. Newer Java (17, 21) causes hard-to-read reflection errors,
so we install Java 8 explicitly rather than trusting whatever Colab ships with.


In [ ]:
%%bash
apt-get -qq update > /dev/null 2>&1
apt-get -qq install -y openjdk-8-jdk-headless > /dev/null 2>&1
echo "Java 8 installed:"
ls -d /usr/lib/jvm/java-8-openjdk-* 2>/dev/null || echo "  !! not found — see Troubleshooting at the end"


### 1.2 · Download Hadoop and Hive

Hadoop is ~700 MB, so this is the slow cell. Colab's network is fast — expect a few minutes.
Each download tries a backup mirror if the first fails.


In [ ]:
import os, subprocess, sys
from pathlib import Path

BASE = Path("/content")
HADOOP_VER, HIVE_VER = "3.3.6", "3.1.3"

HADOOP_URLS = [
    f"https://archive.apache.org/dist/hadoop/common/hadoop-{HADOOP_VER}/hadoop-{HADOOP_VER}.tar.gz",
    f"https://downloads.apache.org/hadoop/common/hadoop-{HADOOP_VER}/hadoop-{HADOOP_VER}.tar.gz",
]
HIVE_URLS = [
    f"https://archive.apache.org/dist/hive/hive-{HIVE_VER}/apache-hive-{HIVE_VER}-bin.tar.gz",
    f"https://downloads.apache.org/hive/hive-{HIVE_VER}/apache-hive-{HIVE_VER}-bin.tar.gz",
]

def fetch(urls, out):
    "Try each mirror until one works."
    out = BASE / out
    if out.exists():
        print(f"  already downloaded: {out.name}"); return out
    for u in urls:
        print(f"  trying {u.split('/')[2]} ...")
        r = subprocess.run(["wget", "-q", "--show-progress", "-O", str(out), u])
        if r.returncode == 0 and out.stat().st_size > 1_000_000:
            print(f"  OK -> {out.name} ({out.stat().st_size/1e6:.0f} MB)"); return out
        out.unlink(missing_ok=True)
    raise RuntimeError(f"All mirrors failed for {out.name}")

os.chdir(BASE)
print("Hadoop:"); h_tar = fetch(HADOOP_URLS, f"hadoop-{HADOOP_VER}.tar.gz")
print("Hive:");   v_tar = fetch(HIVE_URLS,   f"apache-hive-{HIVE_VER}-bin.tar.gz")

print("\nExtracting (about a minute) ...")
if not (BASE / f"hadoop-{HADOOP_VER}").exists():
    subprocess.run(["tar", "-xzf", str(h_tar), "-C", str(BASE)], check=True)
if not (BASE / f"apache-hive-{HIVE_VER}-bin").exists():
    subprocess.run(["tar", "-xzf", str(v_tar), "-C", str(BASE)], check=True)
print("Done.")


### 1.3 · Environment variables + the two classic fixes

Two well-known problems are handled here **before** they can bite:

| Problem | Symptom you would otherwise see | Fix applied |
|---|---|---|
| **Guava version clash** | `NoSuchMethodError: com.google.common.base.Preconditions.checkArgument` | Delete Hive's old `guava-19` jar, copy Hadoop's newer one in |
| **Broken config template** | `Illegal character entity: expansion character (code 0x8 ...)` | Write our own clean minimal `hive-site.xml` instead of copying Hive's template |

We also point Hive at the **local filesystem** (`file:///`) so no HDFS or cluster is needed.


In [ ]:
import glob, shutil, textwrap

# --- JAVA_HOME: find it rather than hard-coding (survives Colab image changes)
java_home = sorted(glob.glob("/usr/lib/jvm/java-8-openjdk-*"))
assert java_home, "Java 8 not found — re-run cell 1.1"
os.environ["JAVA_HOME"]   = java_home[0]
os.environ["HADOOP_HOME"] = str(BASE / f"hadoop-{HADOOP_VER}")
os.environ["HIVE_HOME"]   = str(BASE / f"apache-hive-{HIVE_VER}-bin")
os.environ["PATH"] = (f"{os.environ['HIVE_HOME']}/bin:{os.environ['HADOOP_HOME']}/bin:"
                      f"{os.environ['JAVA_HOME']}/bin:" + os.environ["PATH"])
os.environ["HADOOP_CLIENT_OPTS"] = "-Xmx1g"     # keep Colab RAM happy

# --- FIX 1: guava clash (match by glob so exact versions don't matter)
hive_guava   = glob.glob(f"{os.environ['HIVE_HOME']}/lib/guava-*.jar")
hadoop_guava = glob.glob(f"{os.environ['HADOOP_HOME']}/share/hadoop/hdfs/lib/guava-*.jar")
if hive_guava and hadoop_guava:
    for g in hive_guava:
        os.remove(g)
    shutil.copy(hadoop_guava[0], f"{os.environ['HIVE_HOME']}/lib/")
    print("Guava fix applied:", os.path.basename(hadoop_guava[0]))

# --- FIX 2: our own clean hive-site.xml (avoids the corrupt shipped template)
WAREHOUSE = BASE / "hive_warehouse"; WAREHOUSE.mkdir(exist_ok=True)
SCRATCH   = BASE / "hive_scratch";   SCRATCH.mkdir(exist_ok=True)

def prop(n, v):
    return f"  <property><name>{n}</name><value>{v}</value></property>"

hive_site = "<?xml version=\"1.0\"?>\n<configuration>\n" + "\n".join([
    prop("javax.jdo.option.ConnectionURL", "jdbc:derby:;databaseName=/content/metastore_db;create=true"),
    prop("javax.jdo.option.ConnectionDriverName", "org.apache.derby.jdbc.EmbeddedDriver"),
    prop("hive.metastore.warehouse.dir", f"file://{WAREHOUSE}"),
    prop("hive.exec.scratchdir", str(SCRATCH)),
    prop("fs.defaultFS", "file:///"),                    # no HDFS needed
    prop("mapreduce.framework.name", "local"),           # no YARN needed
    prop("hive.execution.engine", "mr"),
    prop("hive.metastore.schema.verification", "false"),
    prop("hive.metastore.event.db.notification.api.auth", "false"),
    prop("hive.cli.print.header", "true"),               # show column names in output
]) + "\n</configuration>\n"

Path(f"{os.environ['HIVE_HOME']}/conf/hive-site.xml").write_text(hive_site)
print("hive-site.xml written")
print("JAVA_HOME  :", os.environ["JAVA_HOME"])
print("HIVE_HOME  :", os.environ["HIVE_HOME"])


### 1.4 · Initialise the metastore

Hive keeps its **table definitions** (names, columns, types) in a small database called the
**metastore**. We use the built-in Derby database — fine for one user.

> **Concept worth saying out loud:** the metastore holds *only the description* of the table.
> The actual data stays in files. That's why Hive is called **schema-on-read** — you attach a
> table shape to files that already exist, instead of loading data into a database first.


In [ ]:
%%bash
cd /content
rm -rf metastore_db          # clean slate; safe to re-run this cell
$HIVE_HOME/bin/schematool -dbType derby -initSchema 2>&1 | tail -6


### 1.5 · Check it works

If you see `OK` and a list containing `default`, Hive is running.


In [ ]:
%%bash
cd /content
hive -e "SHOW DATABASES;" 2>/dev/null


---
## Section 2 · The data — an Apache web server log

Every web server writes a **log file**: one line per request. This is the classic
*semi-structured* data from Session 1 — it has structure, but it isn't a neat table.

We generate a realistic log in **Apache Combined Log Format** so the notebook is self-contained.
A single line looks like this:

```
203.0.113.5 - - [11/Oct/2025:13:55:36 +0530] "GET /product/shoes HTTP/1.1" 200 5324 "-" "Mozilla/5.0"
```

Reading left to right: **who** asked (IP), **when**, **what** they asked for (method + page),
**what happened** (status code — 200 is fine, 404 is missing, 500 is broken), and **how many bytes** came back.


In [ ]:
import random
from datetime import datetime, timedelta

random.seed(42)                          # everyone gets identical data
LOG_PATH = BASE / "logdata" / "access_log"
LOG_PATH.parent.mkdir(exist_ok=True)

PAGES = ["/", "/product/shoes", "/product/laptop", "/cart", "/checkout",
         "/search", "/offers", "/account", "/product/phone", "/help"]
AGENTS = ["Mozilla/5.0 (Windows NT 10.0)", "Mozilla/5.0 (Macintosh)",
          "Mozilla/5.0 (iPhone)", "Mozilla/5.0 (Android)"]
start = datetime(2025, 10, 11, 0, 0, 0)

lines = []
for i in range(200_000):
    ip     = f"203.0.113.{random.randint(1,250)}"
    ts     = start + timedelta(seconds=random.randint(0, 86_399))
    page   = random.choices(PAGES, weights=[30,14,12,10,7,9,6,5,4,3])[0]
    method = random.choices(["GET","POST"], weights=[92,8])[0]
    status = random.choices([200,404,500,302], weights=[90,5,2,3])[0]
    size   = random.randint(400, 9000) if status == 200 else random.randint(100, 700)
    stamp  = ts.strftime("%d/%b/%Y:%H:%M:%S +0530")
    lines.append(f'{ip} - - [{stamp}] "{method} {page} HTTP/1.1" {status} {size} "-" "{random.choice(AGENTS)}"')

LOG_PATH.write_text("\n".join(lines) + "\n")
print(f"Wrote {len(lines):,} log lines to {LOG_PATH}\n")
print("First 3 lines:")
for l in lines[:3]:
    print("  ", l)


---
## Section 3 · A helper so queries are easy to read

Hive prints a lot of technical logging around each answer. This helper runs a HiveQL query and
shows **only the result**, so the output stays readable in class.

You still write ordinary HiveQL — the helper just tidies the display.


In [ ]:
import subprocess, textwrap, tempfile, os

def hql(query, show_query=True):
    "Run a HiveQL query and print just the results."
    if show_query:
        print("HiveQL:", textwrap.dedent(query).strip(), "\n" + "-"*60)
    # Strip a trailing ';' — this Hive build fails to strip it itself for
    # multi-line statements ("extraneous input ';' expecting EOF"), so we
    # never hand it one.
    clean_query = query.strip()
    if clean_query.endswith(";"):
        clean_query = clean_query[:-1]
    with tempfile.NamedTemporaryFile(mode="w", suffix=".hql", dir="/content",
                                      delete=False) as f:
        f.write(clean_query)
        script_path = f.name
    try:
        r = subprocess.run(["hive", "-f", script_path], cwd="/content",
                           capture_output=True, text=True)
    finally:
        os.remove(script_path)
    if r.stdout.strip():
        print(r.stdout.strip())
    if r.returncode != 0:
        print("\n--- Hive reported a problem ---")
        print("\n".join(r.stderr.strip().splitlines()[-15:]))
    return r.stdout

print("Helper ready. Use:  hql(\"SELECT ...\")")


---
## Section 4 · Create a Hive table over the log file

Here is the idea that makes Hive click:

> **We are not loading the data anywhere. We are laying a table shape *on top of* a file that already exists.**

The log lines aren't comma-separated, so we describe each line with a **pattern** (a regular
expression) that tells Hive which piece is the IP, which is the timestamp, and so on.
That's the `RegexSerDe` below — *SerDe* just means "how to read/write these rows."

You never need to write that pattern from scratch. Read it as: *"this column comes from that
part of the line."*


In [ ]:
hql("DROP TABLE IF EXISTS access_logs;", show_query=False)

create_table = r'''
CREATE EXTERNAL TABLE access_logs (
  ip        STRING,
  log_time  STRING,
  method    STRING,
  page      STRING,
  status    INT,
  bytes     INT,
  agent     STRING
)
ROW FORMAT SERDE 'org.apache.hadoop.hive.serde2.RegexSerDe'
WITH SERDEPROPERTIES (
  "input.regex" = '^(\\S+) \\S+ \\S+ \\[([^\\]]+)\\] "(\\S+) (\\S+) [^"]*" (\\d+) (\\d+) "[^"]*" "([^"]*)"$'
)
STORED AS TEXTFILE
LOCATION '/content/logdata';
'''
hql(create_table, show_query=False)
print("Table created.\n")
hql("DESCRIBE access_logs;")


**`EXTERNAL` matters.** It tells Hive: *the file is not mine — I'm only describing it.*
Drop the table later and the log file stays untouched. That is the normal, safe way to point
Hive at data your organisation already has.


In [ ]:
hql("SELECT * FROM access_logs LIMIT 5;")


---
## Section 5 · Answer real questions with HiveQL

This is the payoff: **these are ordinary SQL queries.** If you have written SQL before, you
already know Hive. If you haven't, read each one as an English sentence.


### Q1 · How much traffic did we get?

In [ ]:
hql("SELECT COUNT(*) AS total_requests FROM access_logs;")


### Q2 · Which pages are most popular?
*"Count the requests for each page, show the busiest first."*

In [ ]:
hql('''
SELECT page, COUNT(*) AS hits
FROM access_logs
GROUP BY page
ORDER BY hits DESC
LIMIT 10;
''')


### Q3 · Is anything broken?
Status **404** = page missing, **500** = server error. A business question hides in here:
*are customers hitting errors on pages that matter?*

In [ ]:
hql('''
SELECT status, COUNT(*) AS requests
FROM access_logs
GROUP BY status
ORDER BY requests DESC;
''')


### Q4 · When is the site busiest?
The timestamp looks like `11/Oct/2025:13:55:36 +0530`. The hour sits at characters 13–14,
so we cut it out with `SUBSTR`. This is very common with log data — the raw field needs a
little shaping before it becomes useful.

In [ ]:
hql('''
SELECT SUBSTR(log_time, 13, 2) AS hour_of_day,
       COUNT(*) AS requests
FROM access_logs
GROUP BY SUBSTR(log_time, 13, 2)
ORDER BY hour_of_day;
''')


### Q5 · Which pages produce the most errors?
Two ideas combined: **filter** to error rows, then **group**. This is exactly the
*filter → group → total → sort* rhythm you already use in Excel.

In [ ]:
hql('''
SELECT page,
       COUNT(*) AS error_hits
FROM access_logs
WHERE status IN (404, 500)
GROUP BY page
ORDER BY error_hits DESC
LIMIT 5;
''')


### Q6 · How much data did we serve?
`SUM` on the bytes column, converted to megabytes — the kind of number an infrastructure
budget conversation actually needs.

In [ ]:
hql('''
SELECT ROUND(SUM(bytes)/1024/1024, 2) AS total_megabytes,
       ROUND(AVG(bytes), 0)           AS avg_bytes_per_request
FROM access_logs
WHERE status = 200;
''')


---
## Section 6 · Your turn

Change **one thing at a time** and re-run. You are not expected to write these from a blank page.

**Exercise 1** — Q2 shows the top 10 pages. Change it to show the top **3**.

**Exercise 2** — Find the **busiest visitors**: group by `ip` instead of `page`.

**Exercise 3** — Q5 looks at 404 and 500 together. Change it to show **only 404s**.

**Exercise 4** — Count how many requests came from **iPhone** users.
*Hint:* `WHERE agent LIKE '%iPhone%'`

**Exercise 5 (interpretation, no code)** — Look at your Q3 output. If 5% of all requests are
404s, what would you tell the e-commerce team to do on Monday morning?


In [ ]:
# Exercise 1 — top 3 pages
hql('''
SELECT page, COUNT(*) AS hits
FROM access_logs
GROUP BY page
ORDER BY hits DESC
LIMIT 10;          -- <-- change 10 to 3
''')


In [ ]:
# Exercise 2 — busiest visitors (fill in the blank)
hql('''
SELECT ____ , COUNT(*) AS hits
FROM access_logs
GROUP BY ____
ORDER BY hits DESC
LIMIT 5;
''')


<details><summary><b>Show solutions</b></summary>

```sql
-- Ex 1: change LIMIT 10 to LIMIT 3

-- Ex 2:
SELECT ip, COUNT(*) AS hits FROM access_logs GROUP BY ip ORDER BY hits DESC LIMIT 5;

-- Ex 3:
SELECT page, COUNT(*) AS error_hits FROM access_logs
WHERE status = 404 GROUP BY page ORDER BY error_hits DESC LIMIT 5;

-- Ex 4:
SELECT COUNT(*) AS iphone_requests FROM access_logs WHERE agent LIKE '%iPhone%';
```
</details>


---
## Section 7 · What you just did

- Installed **real Apache Hive** and pointed it at a raw web-server log.
- Used **schema-on-read**: you described a table *over files that already existed* — you never
  loaded data into a database.
- Answered six genuine business questions in **plain SQL**: traffic, popular pages, errors,
  peak hours, error hotspots, bandwidth.

**The idea to carry forward:** Hive lets you ask SQL questions of files that are far too large
for a spreadsheet — without learning a new language. The queries you wrote today would run
unchanged over billions of log lines on a real cluster. *Only the scale changes.*


---
## Troubleshooting

**"Java 8 not found"** — Colab's base image changed. Try:
`!apt-get install -y openjdk-11-jdk-headless` then edit the glob in 1.3 to `java-11-openjdk-*`.

**Download fails / all mirrors failed** — Check the version still exists at
[archive.apache.org/dist/hive](https://archive.apache.org/dist/hive/) and
[archive.apache.org/dist/hadoop/common](https://archive.apache.org/dist/hadoop/common/),
then update `HADOOP_VER` / `HIVE_VER` in cell 1.2.

**`NoSuchMethodError ... Preconditions.checkArgument`** — the guava fix in 1.3 didn't apply.
Re-run 1.3 and confirm it prints "Guava fix applied".

**`Schema initialization ... already exists`** — re-run cell 1.4; it deletes `metastore_db` first.

**Everything worked, then stopped** — your Colab runtime restarted. Re-run Section 1.

**Queries are slow** — expected. Each Hive query starts a small local job; a few seconds per
query is normal here. On a real cluster the same query runs over vastly more data.
